In [1]:
import os
import re
from glob import glob

# isort: off
import xarray as xr
import pint_xarray
import cf_xarray.units
from pint_xarray import unit_registry as ureg
# isort: on

from utils import (
    PLASTIC_MW,
    PLASTIC_SIZES,
    PLASTIC_SOURCES,
    RESULTS_DIR,
    SIMS_DIR,
    clean_styler,
    convert_molec_to_mass,
    convert_vvdry_to_ugm3,
    spatial_integrate,
)

# Overwrite outputs
OVERWRITE = True

# Use CF as default unit formatting
ureg.default_format = "cf"


def load_sim_output(
    sim_dir: str,
    collection: str,
    source: str = "all",
    chunks: str | dict = "auto",
    reduce_ocean: bool = False,
) -> xr.Dataset:
    """Load raw outputs of GEOS-Chem microplastic simulations

    Args:
        sim_dir: Directory containing the simulation outputs
        collection: One of ["emission", "concentration", "total_deposition",
            "dry_deposition", "wet_loss"]
        source: One of ["all", "ocean", "mmpw", "agricultural", "residential", "road"].
            "all" (default) loads diagnostics for all microplastic sources.
        chunks: Optional dictionary specifying chunks passed to xr.open_mfdataset().
            "auto" (default) sets chunks to a multiple of on-disk chunks.
        reduce_ocean: Whether to reduce ocean microplastics by a factor of 1e-9 (default:
            False). Used for simulations using Fu2023's emission factors, which result in
            ocean microplastic emissions that are orders of magnitude larger than
            emissions from other sources.

    Returns:
        xarray.Dataset including microplastic variable(s) of dimension:
            * [size, time, lat, lon] for "emission" and "dry_deposition" collections
            * [size, time, lev, lat, lon] for all other collections
            * [source, size, time, ...] if source was "all"

    Assumes the simulation outputs are organized in subdirs of sim_dir by source
    [1-ocen, 2-mmpw,  3-agri, 4-resi, 5-road] and timeperiod e.g. 2018-01-01_2019-01-01:

        [sim_dir]
        ├── [source]
        │   ├── [timeperiod]
        │   │   └── OutputDir
        │   │       └── *.nc4
        │   └── ...
        └── ...
    """

    # Collection name -> output files glob
    glob_mapping = dict(
        emission="HEMCO_diagnostics.*.nc",
        concentration="GEOSChem.SpeciesConc.*.nc4",
        total_deposition="GEOSChem.[DW][re][yt][DL]*.nc4",
        dry_deposition="GEOSChem.DryDep.*.nc4",
        wet_loss="GEOSChem.WetLoss*.nc4",
    )

    # Validate collection
    collection = collection.lower()
    if collection not in glob_mapping:
        raise ValueError(
            f"Unrecognized collection '{collection}'. Must be one of {list(glob_mapping)}"
        )

    # If source is None, load and return data for all sources
    source = source.lower()
    if source == "all":
        data = []
        for src in PLASTIC_SOURCES:
            data.append(
                load_sim_output(
                    sim_dir=sim_dir,
                    collection=collection,
                    source=src,
                    chunks=chunks,
                    reduce_ocean=reduce_ocean,
                )
            )
        data = xr.concat(data, dim="source")
        return data

    # Validate source
    if source not in PLASTIC_SOURCES:
        raise ValueError(
            f"Unrecognized source '{source}'. Must be one of {PLASTIC_SOURCES}"
        )

    # List simulation output files
    # e.g. "data/1-ocen/{START DATE}_{END DATE}/OutputDir/HEMCO_diagnostics.*.nc"
    dirnames = {
        x: f"{i + 1}-" + x[0:4].replace("ocea", "ocen")
        for i, x in enumerate(PLASTIC_SOURCES)
    }
    date_glob = "20[12][8901]-[01][0-9]-[0-3][0-9]"
    file_glob = os.path.join(
        sim_dir,
        dirnames[source],
        date_glob + "_" + date_glob,  # simulation period
        "OutputDir",
        glob_mapping[collection],  # diagnostic collection
    )
    paths = sorted(glob(file_glob))
    if not any(paths):
        raise ValueError(f"No {source} {collection} data in {sim_dir}")

    # Load simulation output
    data = xr.open_mfdataset(paths, chunks=chunks)

    # Rename emission variables to conform to GEOS-Chem's standard naming scheme
    # e.g. EmisPST1_OCEN -> Emis_PST1
    # This will facilitate matching on and extracting parts of variable names
    if collection == "emission":
        emis_vars = [v for v in data.variables if v.startswith("EmisPST")]
        for var in emis_vars:
            data[var].attrs.update(
                long_name="Emission flux of species " + data[var].attrs["long_name"][0:4]
            )
        data = data.rename_vars(
            {v: re.sub(r"(PST\d)_[A-Z]+", r"_\1", v) for v in emis_vars}
        )

    # Define variables to reduce by a factor of 1e-9 if reduce_ocean is True
    reduce_vars = [
        "Emis_PST",
        "SpeciesConcVV_PST",
        "DryDep_PST",
        "WetLossConv_PST",
        "WetLossLS_PST",
    ]
    reduce_scale = 1e-9

    # Stack microplastic variables along a new "size" dimension
    # * Each microplastic field (e.g. DryDep) becomes one variable
    # * Some collections include multiple microplastic fields e.g. DryDep and DryDepVel
    size_pattern = r"(\w+_PST)(\d+)"
    size_mapping = {f"PST{i + 1}": x for i, x in enumerate(PLASTIC_SIZES)}
    mp_vars = [v for v in data.variables if re.match(size_pattern, v)]
    if any(mp_vars):
        mp_fields = set(re.sub(size_pattern, r"\1", v) for v in mp_vars)
        for field in sorted(mp_fields):
            # Stack the field variables along a new dimension "size"
            field_vars = sorted(v for v in mp_vars if v.startswith(field))
            size_dim = xr.Variable(
                dims="size",
                data=[
                    size_mapping[re.sub(size_pattern, r"PST\2", v)] for v in field_vars
                ],
                attrs={"long_name": "Aerodynamic diameter", "units": "µm"},
            )
            field_data = (
                data[field_vars].to_dataarray(dim="size").assign_coords(size=size_dim)
            )

            # Possibly reduce ocean microplastics
            if reduce_ocean and source == "ocean" and field in reduce_vars:
                print(f"Scaling ocean {field} by {reduce_scale}")
                with xr.set_options(keep_attrs=True):
                    field_data = reduce_scale * field_data

            # Set attributes
            # * Modify long name
            #   e.g. "Concentration of species PST1" -> "Concentration of microplastics"
            field_data.attrs = data[field_vars[0]].attrs
            field_data.attrs["long_name"] = (
                field_data.attrs["long_name"]
                .replace(" species ", " microplastics ")
                .removesuffix(" PST1")
            )

            # Replace the original field variables with the new field variable
            data = data.drop_vars(field_vars).assign({field: field_data})

    # Add a "source" dimension
    data = data.assign(
        source=xr.Variable(
            dims="source", data=[source], attrs={"long_name": "Emission source"}
        )
    ).set_coords("source")

    return data


def process_sim_output(
    sim_dir: str, label: str, reduce_ocean: bool = False
) -> xr.Dataset:
    """Postprocess outputs of GEOS-Chem microplastic simulations

    Args:
        sim_dir: Directory containing the simulation outputs
        label: Label to identify simulation
        reduce_ocean: Whether to reduce ocean microplastics by a factor of 1e-9 (default:
            False). Used for simulations using Fu2023's emission factors, which result in
            ocean microplastic emissions that are about nine orders of magnitude higher
            than emissions from any other source.

    Returns:
        xarray.Dataset with time-averaged atmospheric microplastic and other variables:
            * emission (kg/km2/year) [source, size, lat, lon]
            * concentration (µg/m3) [source, size, lev, lat, lon]
            * dry_deposition (kg/km2/year) [source, size, lat, lon]
            * wet_loss (kg/km2/year) [source, size, lat, lon]
            * total_deposition (kg/km2/year) [source, size, lat, lon]
            * air_volume (m3) [lev, lat, lon]
            * area: (m2) [lat, lon]
    """

    # Define units for final output
    emis_units = "kg/km2/year"
    conc_units = "µg/m3"
    depo_units = "kg/km2/year"

    # Compute MP emissions
    with xr.set_options(keep_attrs=True):
        emis_outputs = load_sim_output(
            sim_dir=sim_dir, collection="emission", reduce_ocean=reduce_ocean
        )
        emis = emis_outputs["Emis_PST"].pint.quantify().pint.to(emis_units)
        emis.attrs["averaging_method"] = "time-averaged"

    # Compute MP concentration
    with xr.set_options(keep_attrs=True):
        concentration = load_sim_output(
            sim_dir=sim_dir, collection="concentration", reduce_ocean=reduce_ocean
        )

        # Convert concentration mixing ratio to mass per volume
        conc = convert_vvdry_to_ugm3(
            spec=concentration["SpeciesConcVV_PST"],
            spec_mw=PLASTIC_MW,
            air_density=concentration["Met_AIRDEN"],
        ).pint.to(conc_units)

    # Compute MP total deposition
    with xr.set_options(keep_attrs=True):
        wet_outputs = load_sim_output(
            sim_dir=sim_dir, collection="wet_loss", reduce_ocean=reduce_ocean
        )

        # Compute total wet loss (convective + large-scale precipitation)
        wet_loss = (
            (wet_outputs["WetLossConv_PST"] + wet_outputs["WetLossLS_PST"])
            .assign_attrs(long_name="Wet loss flux of soluble microplastics")
            .sum(dim="lev")
            .pint.quantify()
        )

        # Compute wet loss per unit area
        area = wet_outputs["AREA"].pint.quantify()
        wet_loss = (wet_loss / area).pint.to(depo_units)

        # Load dry deposition
        dry_outputs = load_sim_output(
            sim_dir=sim_dir, collection="dry_deposition", reduce_ocean=reduce_ocean
        )

        # Convert dry deposition from (molecules/cm2/s) to (kg/km2/year)
        dry_dep = convert_molec_to_mass(
            spec=dry_outputs["DryDep_PST"], spec_mw=PLASTIC_MW
        ).pint.to(depo_units)

        # Compute total deposition (wet + dry)
        total_dep = (wet_loss + dry_dep).assign_attrs(
            long_name="Total deposition flux of microplastics"
        )

    # Compute dry air volume (m3)
    # This will allow computing atmospheric burden (kg) from concentration (µg/m3)
    # Note: we derive dry air volume from dry air mass + dry air density for consistency
    # with GEOS-Chem's internal conversions, which use dry air mass and density rather
    # than grid cell volume (which is not precisely defined). Deriving dry air volume from
    # dry air mass and density gives values close, but not identical, to the StateMet
    # "Met_AIRVOL" variable, which is computed by multiplying grid cell area by height.
    air_mass = concentration["Met_AD"].isel(source=0, drop=True)
    air_density = concentration["Met_AIRDEN"].isel(source=0, drop=True)
    air_volume = (
        air_mass.pint.quantify({"lev": None}) / air_density.pint.quantify({"lev": None})
    ).assign_attrs(long_name="Dry air volume", averaging_method="time-averaged")

    # Create final dataset
    result = xr.Dataset(
        {
            "emission": emis.pint.dequantify(),
            "concentration": conc.pint.dequantify(),
            "dry_deposition": dry_dep.pint.dequantify(),
            "wet_loss": wet_loss.pint.dequantify(),
            "total_deposition": total_dep.pint.dequantify(),
            "air_volume": air_volume.pint.dequantify(),
            "area": area.isel(source=0).pint.dequantify(),
            "ilev": concentration["ilev"],
            "lat_bnds": concentration["lat_bnds"].isel(source=0),
            "lon_bnds": concentration["lon_bnds"].isel(source=0),
        },
    )

    # Compute mean over study period
    result = result.mean(dim="time", keep_attrs=True)

    # Label
    name = "alternate" if label == "alt" else label
    result = result.assign_attrs(
        description=f"Simulated atmospheric microplastics from {name} configuration",
        label=label,
    )

    # Set order of coordinate variables (cosmetic only; dimension order unchanged)
    result = result.assign_coords(
        {
            "source": result["source"],
            "size": result["size"],
            "lev": result["lev"],
            "lat": result["lat"],
            "lon": result["lon"],
            "ilev": result["ilev"],
            "nb": result["nb"],
        }
    )

    return result


def process_or_load(
    sim_name: str, out_dir: str = RESULTS_DIR, overwrite: bool = False, reduce_ocean=False
) -> xr.Dataset:
    """Postprocess simulation outputs"""

    out_path = os.path.join(out_dir, f"sim_{sim_name}.nc")
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return xr.open_dataset(out_path, chunks={})

    # Postprocess outputs
    sim_dir = os.path.join(SIMS_DIR, sim_name)
    sim = process_sim_output(sim_dir=sim_dir, label=sim_name, reduce_ocean=reduce_ocean)

    # Save as netCDF
    sim.to_netcdf(out_path)
    print(f"-> {out_path}")

    # Load with data chunks for consistency
    return xr.open_dataset(out_path, chunks={})


Introduction
------------

This notebook postprocesses the raw outputs of GEOS-Chem atmospheric microplastic cycling simulations. We:

* Aggregate outputs for all microplastic sources in a single netCDF file
* Compute time-averaged mean emissions, concentration, and deposition over the three-year study period (2018-2020)
* Sum wet deposition over all model levels
* Convert concentration from mol/mol dry air to µg/m^3^

Process *alternate* simulation
------------------------------

This simulation uses the configuration of @Fu2023. It simulates atmospheric microplastic emissions from five sources:

- 233 Gg/year from oceans
- 0.5 Gg/year from mismanaged plastic waste
- 4 Gg/year from agriculture
- 0.1 Gg/year from residences
- 81 Gg/year from roads

The mass is split equally between the six tracer sizes (diameter 0.3, 2.5, 7, 15, 35, 70 µm), except for the 0.3 µm ocean tracer whose emissions are about 1/60 that of the other ocean tracers. This is because the 0.3 µm tracer is computed from accumulation-mode sea salt emissions, whose mass is much less than that of the coarse-mode sea salt emissions that are used to compute the other tracers.

In [2]:
alt = process_or_load(sim_name="alt", reduce_ocean=True, overwrite=OVERWRITE)

Scaling ocean Emis_PST by 1e-09
Scaling ocean SpeciesConcVV_PST by 1e-09
Scaling ocean WetLossConv_PST by 1e-09
Scaling ocean WetLossLS_PST by 1e-09
Scaling ocean DryDep_PST by 1e-09
-> results/sim_alt.nc


### Processed outputs

In [3]:
print(alt)

<xarray.Dataset> Size: 240MB
Dimensions:           (source: 5, size: 6, lat: 91, lon: 144, lev: 72, nb: 2,
                       ilev: 73)
Coordinates:
  * source            (source) <U12 240B 'ocean' 'mmpw' ... 'residential' 'road'
  * size              (size) float64 48B 0.3 2.5 7.0 15.0 35.0 70.0
  * lev               (lev) float64 576B 0.9925 0.9775 ... 2.635e-05 1.5e-05
  * lat               (lat) float64 728B -89.5 -88.0 -86.0 ... 86.0 88.0 89.5
  * lon               (lon) float64 1kB -180.0 -177.5 -175.0 ... 175.0 177.5
  * ilev              (ilev) float64 584B 1.0 0.985 0.97 ... 2e-05 1e-05
  * nb                (nb) int64 16B 0 1
Data variables:
    emission          (source, size, lat, lon) float32 2MB dask.array<chunksize=(5, 6, 91, 144), meta=np.ndarray>
    concentration     (source, size, lev, lat, lon) float64 226MB dask.array<chunksize=(5, 6, 72, 91, 144), meta=np.ndarray>
    dry_deposition    (source, size, lat, lon) float64 3MB dask.array<chunksize=(5, 6, 91, 144), 

In [4]:
tmp = spatial_integrate(alt, varname="emission")
clean_styler(
    tmp.to_dataframe()["emission"]
    .unstack("source")
    .style.set_caption(f"Global emissions [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=3)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.768,0.082,0.727,0.024,13.526
2.5,46.507,0.082,0.727,0.024,13.526
7.0,46.507,0.082,0.727,0.024,13.526
15.0,46.507,0.082,0.727,0.024,13.526
35.0,46.507,0.082,0.727,0.024,13.526
70.0,46.507,0.082,0.727,0.024,13.526


In [5]:
tmp = spatial_integrate(alt, varname="concentration")
clean_styler(
    tmp.to_dataframe()["burden"]
    .unstack("source")
    .style.set_caption(f"Atmospheric burden [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=5)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.01160,0.00281,0.02322,0.00057,0.34224
2.5,0.64729,0.00238,0.01864,0.00044,0.26427
7.0,0.10889,0.00111,0.00683,0.00011,0.06868
15.0,0.09472,0.00100,0.00589,0.00010,0.06051
35.0,0.07103,0.00087,0.00482,0.00008,0.05007
70.0,0.06258,0.00081,0.00427,0.00007,0.04386


In [6]:
tmp = spatial_integrate(alt, varname="total_deposition")
clean_styler(
    tmp.to_dataframe()["total_deposition"]
    .unstack("source")
    .style.set_caption(f"Global deposition [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=3)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.771,0.083,0.734,0.024,13.580
2.5,46.670,0.083,0.733,0.024,13.568
7.0,46.435,0.082,0.729,0.024,13.515
15.0,46.444,0.082,0.729,0.024,13.516
35.0,46.454,0.082,0.728,0.024,13.516
70.0,46.457,0.082,0.728,0.024,13.517


Process *main* simulation
-------------------------

This simulation emits approximately 1 Gg/yr of microplastics from each source (oceans, mismanaged plastic waste, agriculture, residences, roads). The emissions follow a power law particle size distribution with slope of -2.8. This is the median slope of atmospheric microplastic observations in MPsizeBase [@Sonke2025]. We compute this value in [size-harmonize-obs.ipynb](./size-harmonize-obs.ipynb). The resulting mass distribution over the tracer sizes is:

- 0.3 µm = 0.3%
- 2.5 µm = 2.4%
- 7 µm = 3%
- 15 µm = 13%
- 35 µm = 25%
- 70 µm = 56%

In [7]:
main = process_or_load(sim_name="main", reduce_ocean=False, overwrite=OVERWRITE)

-> results/sim_main.nc


### Processed outputs

In [8]:
print(main)

<xarray.Dataset> Size: 240MB
Dimensions:           (source: 5, size: 6, lat: 91, lon: 144, lev: 72, nb: 2,
                       ilev: 73)
Coordinates:
  * source            (source) <U12 240B 'ocean' 'mmpw' ... 'residential' 'road'
  * size              (size) float64 48B 0.3 2.5 7.0 15.0 35.0 70.0
  * lev               (lev) float64 576B 0.9925 0.9775 ... 2.635e-05 1.5e-05
  * lat               (lat) float64 728B -89.5 -88.0 -86.0 ... 86.0 88.0 89.5
  * lon               (lon) float64 1kB -180.0 -177.5 -175.0 ... 175.0 177.5
  * ilev              (ilev) float64 584B 1.0 0.985 0.97 ... 2e-05 1e-05
  * nb                (nb) int64 16B 0 1
Data variables:
    emission          (source, size, lat, lon) float32 2MB dask.array<chunksize=(5, 6, 91, 144), meta=np.ndarray>
    concentration     (source, size, lev, lat, lon) float64 226MB dask.array<chunksize=(5, 6, 72, 91, 144), meta=np.ndarray>
    dry_deposition    (source, size, lat, lon) float64 3MB dask.array<chunksize=(5, 6, 91, 144), 

In [9]:
tmp = spatial_integrate(main, varname="emission")
clean_styler(
    tmp.to_dataframe()["emission"]
    .unstack("source")
    .style.set_caption(f"Global emissions [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=3)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.005,0.005,0.005,0.005,0.005
2.5,0.024,0.024,0.024,0.024,0.024
7.0,0.030,0.030,0.030,0.030,0.030
15.0,0.130,0.132,0.132,0.130,0.130
35.0,0.252,0.256,0.255,0.252,0.252
70.0,0.558,0.565,0.564,0.558,0.558


In [10]:
tmp = spatial_integrate(main, varname="concentration")
clean_styler(
    tmp.to_dataframe()["burden"]
    .unstack("source")
    .style.set_caption(f"Atmospheric burden [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=5)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.00008,0.00018,0.00017,0.00012,0.00013
2.5,0.00033,0.00070,0.00062,0.00044,0.00047
7.0,0.00007,0.00041,0.00028,0.00014,0.00015
15.0,0.00027,0.00160,0.00107,0.00054,0.00058
35.0,0.00039,0.00271,0.00169,0.00084,0.00093
70.0,0.00075,0.00555,0.00330,0.00157,0.00181


In [11]:
tmp = spatial_integrate(main, varname="total_deposition")
clean_styler(
    tmp.to_dataframe()["total_deposition"]
    .unstack("source")
    .style.set_caption(f"Global deposition [{tmp.units}]")
    .format_index(precision=1)
    .format(precision=3)
)

source,ocean,mmpw,agricultural,residential,road
size,,,,,
0.3,0.005,0.005,0.005,0.005,0.005
2.5,0.024,0.024,0.024,0.024,0.024
7.0,0.030,0.030,0.030,0.030,0.030
15.0,0.130,0.132,0.132,0.130,0.130
35.0,0.252,0.256,0.255,0.252,0.252
70.0,0.557,0.565,0.564,0.558,0.558
